<a href="https://colab.research.google.com/github/ishikabeniwal/CSET343/blob/main/Assignment5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# LAB ASSIGNMENT - 5
# MULTICLASS CLASSIFICATION
# Dataset: Dermatology
# ============================================

# -----------------------------
# 1. Import Libraries
# -----------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    roc_auc_score
)

from sklearn.preprocessing import label_binarize

In [ ]:
# -----------------------------
# 2. Load Dataset
# -----------------------------

# Upload dermatology.data in Google Colab
from google.colab import files

uploaded = files.upload()

# Load dataset
df = pd.read_csv("dermatology.data", header=None)

print("Dataset Shape:", df.shape)
df.head()

In [ ]:
# -----------------------------
# 3. Assign Column Names
# -----------------------------

# Dermatology dataset has 34 features + 1 target
columns = [f"feature_{i}" for i in range(1, 35)] + ["Class"]

df.columns = columns

print(df.head())

In [ ]:
# -----------------------------
# 4. Dataset Information
# -----------------------------

print(df.info())

In [ ]:
# -----------------------------
# 5. Summary Statistics
# -----------------------------

print(df.describe())

In [ ]:
# Median
print("Median:")
print(df.median(numeric_only=True))

# Standard deviation
print("\nStandard Deviation:")
print(df.std(numeric_only=True))

In [ ]:
# -----------------------------
# 6. Missing Values
# -----------------------------

print("Missing values:")
print(df.isnull().sum())

# Convert '?' into NaN
df = df.replace("?", np.nan)

print("\nMissing values after replacing '?':")
print(df.isnull().sum())

In [ ]:
# Convert all columns to numeric
df = df.apply(pd.to_numeric)

# Check missing values again
print(df.isnull().sum())

In [ ]:
# -----------------------------
# 7. Handle Missing Values
# -----------------------------

# Fill missing values with median
for column in df.columns:
    if column != "Class":
        df[column] = df[column].fillna(df[column].median())

print("Missing values after handling:")
print(df.isnull().sum())

In [ ]:
# -----------------------------
# 8. Check Duplicates
# -----------------------------

print("Number of duplicate rows:", df.duplicated().sum())

# Remove duplicates
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

In [ ]:
# -----------------------------
# 9. Histograms
# -----------------------------

df.iloc[:, :5].hist(figsize=(12, 8))

plt.suptitle("Histograms of Key Features")
plt.show()

In [ ]:
# -----------------------------
# 10. Box Plots
# -----------------------------

plt.figure(figsize=(15, 6))

sns.boxplot(data=df.iloc[:, :10])

plt.xticks(rotation=45)
plt.title("Box Plot of Key Features")
plt.show()

In [ ]:
# -----------------------------
# 11. Correlation Heatmap
# -----------------------------

plt.figure(figsize=(15, 12))

sns.heatmap(
    df.corr(),
    cmap="coolwarm",
    annot=False
)

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# -----------------------------
# 12. Class Distribution
# -----------------------------

plt.figure(figsize=(8, 5))

sns.countplot(x="Class", data=df)

plt.title("Class Distribution")
plt.xlabel("Disease Class")
plt.ylabel("Number of Samples")
plt.show()

print(df["Class"].value_counts())

In [ ]:
# -----------------------------
# 13. Separate Features and Target
# -----------------------------

X = df.drop("Class", axis=1)
y = df["Class"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# -----------------------------
# 14. Encode Target Variable
# -----------------------------

encoder = LabelEncoder()

y = encoder.fit_transform(y)

print("Encoded classes:")
print(np.unique(y))

In [ ]:
# -----------------------------
# 15. Train-Test Split
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
# -----------------------------
# 16. Standardization
# -----------------------------

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Standardization completed.")

In [ ]:
# -----------------------------
# 17. Create Models
# -----------------------------

models = {
    "KNN": KNeighborsClassifier(n_neighbors=5),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=1000
    ),

    "SVM": SVC(
        probability=True,
        random_state=42
    )
}

In [ ]:
# -----------------------------
# 18. Train Models
# -----------------------------

trained_models = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    trained_models[name] = model

    print(name, "trained successfully")

In [ ]:
# -----------------------------
# 19. Evaluate Models
# -----------------------------

results = []

for name, model in trained_models.items():

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted"
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted"
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted"
    )

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

results_df

In [ ]:
# -----------------------------
# 20. Classification Report
# -----------------------------

for name, model in trained_models.items():

    y_pred = model.predict(X_test)

    print("\n================================")
    print(name)
    print("================================")

    print(
        classification_report(
            y_test,
            y_pred
        )
    )

In [ ]:
# -----------------------------
# 21. Confusion Matrix
# -----------------------------

for name, model in trained_models.items():

    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(7, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues"
    )

    plt.title("Confusion Matrix - " + name)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.show()

In [ ]:
# -----------------------------
# 22. ROC Curve and AUC
# -----------------------------

# Convert target to binary format
n_classes = len(np.unique(y_test))

y_test_binary = label_binarize(
    y_test,
    classes=np.arange(n_classes)
)

plt.figure(figsize=(10, 7))

for name, model in trained_models.items():

    # Get probability predictions
    y_score = model.predict_proba(X_test)

    # Calculate macro AUC
    auc_score = roc_auc_score(
        y_test_binary,
        y_score,
        multi_class="ovr",
        average="macro"
    )

    # Micro-average ROC
    fpr = dict()
    tpr = dict()

    for i in range(n_classes):

        fpr[i], tpr[i], _ = roc_curve(
            y_test_binary[:, i],
            y_score[:, i]
        )

    # Calculate average curve
    all_fpr = np.unique(
        np.concatenate(
            [fpr[i] for i in range(n_classes)]
        )
    )

    mean_tpr = np.zeros_like(all_fpr)

    for i in range(n_classes):

        mean_tpr += np.interp(
            all_fpr,
            fpr[i],
            tpr[i]
        )

    mean_tpr /= n_classes

    plt.plot(
        all_fpr,
        mean_tpr,
        label=f"{name} (AUC = {auc_score:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("Multiclass ROC Curve")

plt.legend()

plt.show()

In [ ]:
# -----------------------------
# 23. Final Model Comparison
# -----------------------------

print("Final Model Comparison")

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

print(results_df)

In [ ]:
# -----------------------------
# 24. Best Model
# -----------------------------

best_model = results_df.iloc[0]

print("Best Model:", best_model["Model"])
print("Accuracy:", best_model["Accuracy"])
print("Precision:", best_model["Precision"])
print("Recall:", best_model["Recall"])
print("F1 Score:", best_model["F1 Score"])